# T5Gemma with CTranslate2

This notebook demonstrates how to use T5Gemma models with CTranslate2 for efficient inference.

T5Gemma combines the encoder-decoder architecture of T5 with Gemma 2 improvements:
- Grouped Query Attention (GQA)
- Rotary Position Embeddings (RoPE)
- GeGLU activation
- RMSNorm
- Interleaved local/global attention

## Installation

Install CTranslate2 with T5Gemma support and dependencies:

In [ ]:
# Install CTranslate2 from the development branch with T5Gemma support
!pip install -q git+https://github.com/jncraton/CTranslate2.git@copilot/support-t5gemma-architecture

# Install transformers with T5Gemma support (requires recent version)
!pip install -q 'transformers>=4.50.0' torch sentencepiece

## Verify Installation

In [ ]:
import ctranslate2
from ctranslate2.converters.transformers import _MODEL_LOADERS

# Check if T5Gemma support is available
if "T5GemmaConfig" in _MODEL_LOADERS:
    print("✓ T5Gemma support is available!")
    print(f"  Loader: {_MODEL_LOADERS['T5GemmaConfig'].__class__.__name__}")
    print(f"  Architecture: {_MODEL_LOADERS['T5GemmaConfig'].architecture_name}")
else:
    print("✗ T5Gemma support is not available")

## Convert T5Gemma Model

We'll use the `harshaljanjani/tiny-t5gemma-test` model for demonstration (it's small and quick to download):

In [ ]:
model_name = "harshaljanjani/tiny-t5gemma-test"
output_dir = "ct2_t5gemma_model"

print(f"Converting {model_name}...")

converter = ctranslate2.converters.TransformersConverter(
    model_name,
    trust_remote_code=True
)

converted_model_path = converter.convert(output_dir)
print(f"✓ Model converted successfully to: {converted_model_path}")

## Load Tokenizer

Load the tokenizer from the original model:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("✓ Tokenizer loaded")

## Run Inference

Now let's test the converted model with some example inputs:

In [ ]:
# Load the converted model
translator = ctranslate2.Translator(converted_model_path)

# Example 1: Translation
source_text = "translate English to German: The house is wonderful."
source_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(source_text))

results = translator.translate_batch([source_tokens])

# Decode the result
target_tokens = results[0].hypotheses[0]
target_text = tokenizer.decode(
    tokenizer.convert_tokens_to_ids(target_tokens),
    skip_special_tokens=True
)

print("Source:", source_text)
print("Target:", target_text)

## Batch Inference

CTranslate2 efficiently handles batch processing:

In [ ]:
# Multiple translation examples
source_texts = [
    "translate English to French: Hello world",
    "translate English to German: Good morning",
    "translate English to Spanish: Thank you",
]

# Tokenize all inputs
source_tokens_batch = [
    tokenizer.convert_ids_to_tokens(tokenizer.encode(text))
    for text in source_texts
]

# Run batch translation
results = translator.translate_batch(source_tokens_batch)

# Display results
print("\nBatch Translation Results:")
print("=" * 60)
for source, result in zip(source_texts, results):
    target_tokens = result.hypotheses[0]
    target_text = tokenizer.decode(
        tokenizer.convert_tokens_to_ids(target_tokens),
        skip_special_tokens=True
    )
    print(f"Source: {source}")
    print(f"Target: {target_text}")
    print("-" * 60)

## Advanced: Beam Search and Sampling

CTranslate2 supports various decoding strategies:

In [ ]:
source_text = "translate English to German: The house is wonderful."
source_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(source_text))

# Beam search with multiple hypotheses
results = translator.translate_batch(
    [source_tokens],
    beam_size=5,  # Generate 5 hypotheses
    num_hypotheses=3,  # Return top 3
    return_scores=True  # Include scores
)

print("\nBeam Search Results:")
print("=" * 60)
for i, (hypothesis, score) in enumerate(zip(results[0].hypotheses, results[0].scores)):
    target_text = tokenizer.decode(
        tokenizer.convert_tokens_to_ids(hypothesis),
        skip_special_tokens=True
    )
    print(f"Hypothesis {i+1} (score: {score:.4f}): {target_text}")

## Model Information

Get information about the converted model:

In [ ]:
import os

# Check model size
model_size = sum(
    os.path.getsize(os.path.join(dirpath, filename))
    for dirpath, dirnames, filenames in os.walk(converted_model_path)
    for filename in filenames
)

print(f"\nModel Information:")
print(f"  Path: {converted_model_path}")
print(f"  Size: {model_size / (1024**2):.2f} MB")
print(f"  Device: {translator.device}")
print(f"  Device Index: {translator.device_index}")
print(f"  Compute Type: {translator.compute_type}")

## Clean Up

Optionally remove the converted model to free up space:

In [ ]:
# Uncomment to remove the converted model
# import shutil
# shutil.rmtree(converted_model_path)
# print(f"Removed {converted_model_path}")